In [ ]:
from pyspark.sql.functions import col, current_timestamp

In [ ]:
dbutils.widgets.text("catalog", "olist_project_dev")

dbutils.widgets.text("bronze_schema", "olist_bronze")
dbutils.widgets.text("raw_olist_customers_table", "olist_customers")

dbutils.widgets.text("silver_schema", "olist_silver")
dbutils.widgets.text("customers_table", "customers_silver")

In [ ]:
catalog = dbutils.widgets.get("catalog")

bronze_schema = dbutils.widgets.get("bronze_schema")
raw_olist_customers_table_name = dbutils.widgets.get("raw_olist_customers_table")

silver_schema = dbutils.widgets.get("silver_schema")
customers_table_name = dbutils.widgets.get("customers_table")

In [ ]:
raw_olist_customers_df = spark.table(f"{catalog}.{bronze_schema}.{raw_olist_customers_table_name}")

In [ ]:
if not spark.catalog.tableExists(f"{catalog}.{silver_schema}.{customers_table_name}"):
    spark.sql(
        f"""
        CREATE TABLE {catalog}.{silver_schema}.{customers_table_name} (
            customerId STRING,
            customerUniqueId STRING,
            customerZipCodePrefix STRING,
            customerCity STRING,
            customerState STRING,
            processedTimestamp TIMESTAMP
        )
        TBLPROPERTIES (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact' = 'true'
        )
        """
    )

In [ ]:
customers_silver_df = (
    raw_olist_customers_df
    .where(
        (col("customer_id").rlike("^[0-9a-fA-F]{32}$")) & (col("customer_unique_id").rlike("^[0-9a-fA-F]{32}$"))
    )
    .select(
        col("customer_id").cast("string").alias("customerId"),
        col("customer_unique_id").cast("string").alias("customerUniqueId"),
        col("customer_zip_code_prefix").cast("string").alias("customerZipCodePrefix"),
        col("customer_city").cast("string").alias("customerCity"),
        col("customer_state").cast("string").alias("customerState")
    )
    .withColumn("processedTimestamp", current_timestamp()) 
)

In [ ]:
customers_silver_df.createOrReplaceTempView("customers_silver_view")

spark.sql(f"""
    MERGE INTO {catalog}.{silver_schema}.{customers_table_name} AS target
    USING customers_silver_view AS source
    ON target.customerId = source.customerId
    WHEN MATCHED THEN
        UPDATE SET
            target.customerUniqueId = source.customerUniqueId,
            target.customerZipCodePrefix = source.customerZipCodePrefix,
            target.customerCity = source.customerCity,
            target.customerState = source.customerState
    WHEN NOT MATCHED THEN
        INSERT *
    """)